## <클레블랜드 뮤지엄 데이터 수집>
- 흉배는 API 활용, 그 외 나머지는 직접 수집 예정
- API 설명
   - https://openaccess-api.clevelandart.org/

### 1. 필요한 것들 불러오기

In [9]:
import os
import time
import requests
import pandas as pd
 
OUTPUT_EXCEL = "../data/cma_hyungbae.xlsx"
HEADERS = {"User-Agent": "aks-digital-humanities-research/1.0"}

### 2. API 호출해서 원하는 정보 얻기

In [13]:
MUSEUM_CODE = "CMA"
CATEGORY = "흉배"
CATEGORY_LETTER = "H"
KEYWORD = "rank badge"

rows = []
serial_counter = {}
failed_ids = []

def next_temp_id(museum_code):
    n = serial_counter.get(museum_code, 0) + 1
    serial_counter[museum_code] = n
    return f"Y{museum_code}{CATEGORY_LETTER}{n:02d}"

def pick_url(img_obj, prefer="print"):
    if not img_obj:
        return None
    return (img_obj.get(prefer) or img_obj.get("web") or img_obj.get("full") or {}).get("url")

def fetch_cma(keyword=KEYWORD, page_size=100):
    base = "https://openaccess-api.clevelandart.org/api/artworks/"
    skip = 0
    while True:
        params = {"q": keyword, "limit": page_size, "skip": skip}
        resp = requests.get(base, params=params, headers=HEADERS, timeout=20)
        resp.raise_for_status()
        payload = resp.json()
        data = payload.get("data") or []
        if not data:
            break

        for item in data:
            temp_id = next_temp_id(MUSEUM_CODE)

            main_url = pick_url(item.get("images"))
            alt_urls = [pick_url(a) for a in (item.get("alternate_images") or [])]
            all_images = [u for u in [main_url] + alt_urls if u]

            culture = item.get("culture")
            culture_str = "; ".join(culture) if isinstance(culture, list) else culture

            creators = item.get("creators") or []
            creator_names = "; ".join(c.get("description", "") for c in creators if c.get("description"))

            rows.append({
                "임시ID": temp_id,
                "분류": CATEGORY,
                "소장처": "클리블랜드 미술관",
                "소장처유물번호": item.get("accession_number"),
                "한글명": "",
                "한자명": "",
                "영어명": item.get("title"),
                "URL": item.get("url"),
                "searched_keyword": keyword,
                "culture": culture_str,
                "creators": creator_names,
                "date": item.get("creation_date"),
                "date_text": item.get("date_text"),
                "technique": item.get("technique"),
                "department": item.get("department"),
                "type": item.get("type"),
                "measurements": item.get("measurements"),
                "credit_line": item.get("creditline"),
                "current_location": item.get("current_location"),
                "conservation_statement": item.get("conservation_statement"),
                "description": item.get("description"),
                "tombstone": item.get("tombstone"),
                "image_url": main_url,
                "image_urls": "; ".join(all_images),
                "is_public_domain": item.get("share_license_status") == "CC0",
                "license_note": item.get("share_license_status"),
            })

        if len(data) < page_size:
            break
        skip += page_size
        time.sleep(0.3)

fetch_cma()
print(f"{len(rows)}건 수집 완료")

17건 수집 완료


### 3. 데이터 프레임 -> 엑셀

In [14]:
df = pd.DataFrame(rows)
master_cols = ["임시ID", "분류", "소장처", "소장처유물번호", "한글명", "한자명", "영어명",
               "URL", "searched_keyword", "culture", "creators", "date", "date_text", "technique",
               "department", "type", "measurements", "credit_line", "current_location",
               "conservation_statement", "description", "tombstone",
               "image_url", "image_urls", "is_public_domain", "license_note"]
other_cols = [c for c in df.columns if c not in master_cols]
df = df[master_cols + other_cols]

os.makedirs("../data", exist_ok=True)

pure_hyungbae = df[df["영어명"].str.contains("rank badge", case=False, na=False)]
pure_hyungbae.to_excel(OUTPUT_EXCEL, index=False)

print(pure_hyungbae.shape)
pure_hyungbae.head()

(10, 26)


,임시ID,분류,소장처,소장처유물번호,한글명,한자명,영어명,URL,searched_keyword,culture,...,measurements,credit_line,current_location,conservation_statement,description,tombstone,image_url,image_urls,is_public_domain,license_note
0,YCMAH01,흉배,클리블랜드 미술관,1948.69,,,Rank Badge (buzi),https://clevelandart.org/art/1948.69,rank badge,"China, Qing dynasty (1644–1911), Qianlong reig...",...,Overall: 31.8 x 30.5 cm (12 1/2 x 12 in.),Purchase from the J. H. Wade Fund,NaN,NaN,Rank badges (also called rank insignia or Mand...,"Rank Badge (buzi), 1736–95. China, Qing dynast...",https://openaccess-cdn.clevelandart.org/1948.6...,https://openaccess-cdn.clevelandart.org/1948.6...,True,CC0
1,YCMAH02,흉배,클리블랜드 미술관,1985.32,,,Rank Badge (buzi),https://clevelandart.org/art/1985.32,rank badge,"China, Ming dynasty (1368–1644)",...,Overall: 32 x 36.1 cm (12 5/8 x 14 3/16 in.),Dudley P. Allen Fund,NaN,NaN,Rank badges (also called rank insignia or Mand...,"Rank Badge (buzi), c. 1500–1550. China, Ming d...",https://openaccess-cdn.clevelandart.org/1985.3...,https://openaccess-cdn.clevelandart.org/1985.3...,True,CC0
2,YCMAH03,흉배,클리블랜드 미술관,1948.72,,,Rank Badge (buzi),https://clevelandart.org/art/1948.72,rank badge,"China, Qing dynasty (1644–1911), Qianlong reig...",...,Overall: 27.3 x 26 cm (10 3/4 x 10 1/4 in.),Purchase from the J. H. Wade Fund,NaN,NaN,Rank badges (also called rank insignia or Mand...,"Rank Badge (buzi), 1736–95. China, Qing dynast...",https://openaccess-cdn.clevelandart.org/1948.7...,https://openaccess-cdn.clevelandart.org/1948.7...,True,CC0
3,YCMAH04,흉배,클리블랜드 미술관,2019.78.1,,,Rank Badge with Single Crane Motif,https://clevelandart.org/art/2019.78.1,rank badge,"Korea, Joseon dynasty (1392–1910)",...,25.4 x 25.4 cm (10 x 10 in.),Alma Kroeger Fund,NaN,NaN,"Traditionally, square silk badges with embroid...","Rank Badge with Single Crane Motif (단학흉배), 180...",https://openaccess-cdn.clevelandart.org/2019.7...,https://openaccess-cdn.clevelandart.org/2019.7...,True,CC0
4,YCMAH05,흉배,클리블랜드 미술관,2019.78.2,,,Rank Badge with Single Crane Motif,https://clevelandart.org/art/2019.78.2,rank badge,"Korea, Joseon dynasty (1392–1910)",...,25.4 x 25.4 cm (10 x 10 in.),Alma Kroeger Fund,NaN,NaN,"Traditionally, square silk badges with embroid...","Rank Badge with Single Crane Motif (단학흉배), 180...",https://openaccess-cdn.clevelandart.org/2019.7...,https://openaccess-cdn.clevelandart.org/2019.7...,True,CC0


### 4. 이미지

In [15]:
os.makedirs("../image/hyungbae", exist_ok=True)

pure_hyungbae = pure_hyungbae.reset_index(drop=True)

for i, row in pure_hyungbae.iterrows():
    temp_id = row["임시ID"]
    raw = row["image_urls"]

    if pd.isna(raw) or str(raw).strip() == "":
        continue

    image_urls = [u for u in str(raw).split("; ") if u.strip()]

    for idx, image_url in enumerate(image_urls):
        suffix = "" if len(image_urls) == 1 else f"-{idx + 1}"
        filepath = f"../image/hyungbae/{temp_id}{suffix}.jpg"

        resp = requests.get(image_url, headers=HEADERS, timeout=30)
        with open(filepath, "wb") as f:
            f.write(resp.content)

    if (i + 1) % 20 == 0:
        print(f"{i + 1} / {len(pure_hyungbae)} 완료")

    time.sleep(0.5)